# Get scraped scotus data

We scraped the scotus docket data for all dockets between 2001 and 2024, to the extent they are available on the scotus website.

This notebook outlines the steps undertook to parse relevant information from the scraped pages for linking the appellate chain (scotus to the lower court). The final output is scotus_data.csv

# Import libraries

In [1]:
import os

import numpy as np
import pandas as pd

from parse_utils import parse_scotus_file

# Use the helper functions to get the data & clean as needed

In [2]:
%%time

root_folder = "scotus_dockets/scraped_A/"
count = 0
jsons = []

for dirpath, dirnames, filenames in os.walk(root_folder):
    # Clean up subdirectories
    dirnames[:] = [d for d in dirnames if not d.endswith('.ipynb_checkpoints') and not d.endswith('.DS_Store')]

    # Skip files ending with .ipynb_checkpoints
    filenames = [f for f in filenames if not f.endswith('.ipynb_checkpoints') and not f.endswith('.DS_Store')]
    if not filenames:
        continue

    # Get current folder name
    folder_name = os.path.basename(dirpath)
    json_filename = f"data/scraped_scotus_A/{folder_name}.json"
    print("--- Processing folder:", folder_name)

    results = {}
    for filename in filenames:
        file_path = os.path.join(dirpath, filename)
        results[filename] = parse_scotus_file(file_path)
        count += 1

    df = pd.DataFrame.from_dict(results, orient='index').reset_index().rename(columns={'index': 'filename'})
    df.to_json(json_filename)
    jsons.append(f"{folder_name}.json")
    print(f"Saved results to: {json_filename}")

print(f"\nProcessed {count} files and saved JSONs: {', '.join(sorted(jsons))}")

--- Processing folder: 03
Saved results to: data/scraped_scotus_A/03.json
--- Processing folder: 04
Saved results to: data/scraped_scotus_A/04.json
--- Processing folder: 05
Saved results to: data/scraped_scotus_A/05.json
--- Processing folder: 02
Saved results to: data/scraped_scotus_A/02.json
--- Processing folder: 20
Saved results to: data/scraped_scotus_A/20.json
--- Processing folder: 18
Saved results to: data/scraped_scotus_A/18.json
--- Processing folder: 11
Saved results to: data/scraped_scotus_A/11.json
--- Processing folder: 16
Saved results to: data/scraped_scotus_A/16.json
--- Processing folder: 17
Saved results to: data/scraped_scotus_A/17.json
--- Processing folder: 10
Saved results to: data/scraped_scotus_A/10.json
--- Processing folder: 19
Saved results to: data/scraped_scotus_A/19.json
--- Processing folder: 21
Saved results to: data/scraped_scotus_A/21.json
--- Processing folder: 07
Saved results to: data/scraped_scotus_A/07.json
--- Processing folder: 09
Saved result

# Combine the individual csvs for each year into one large csv

In [3]:
df_list = []

for json_file in jsons:
    df = pd.read_json(f"data/scraped_scotus_A/{json_file}")
    df_list.append(df)

# Combine all DataFrames
scotus_df = pd.concat(df_list, ignore_index=True)

# Rename the case_number column
scotus_df = scotus_df.rename(columns={'case_number': 'docket_number'})

# Get the year and case id for the particular year
scotus_df["year"] = scotus_df["docket_number"].str.split("A").str[0]
scotus_df["case_num"] = scotus_df["docket_number"].str.split("A").str[1].astype(int)

len(scotus_df)

25831

In [4]:
scotus_df.head()

,filename,docket_number,docket_date,case_title,lower_court,lower_court_case_numbers_raw,lower_court_case_numbers,lower_court_decision_date,lower_court_rehearing_denied_date,year,case_num
0,03A517.htm,03A517,NaN,"Randy Kailey, Applicant v. Colorado",Court of Appeals of Colorado,"(01CA1378, 01CA2294)","01CA1378, 01CA2294",NaN,NaN,03,517
1,03A271.htm,03A271,NaN,"Christie M. Browne, Applicant v. John D. Ashcr...",United States Court of Appeals for the Fifth C...,(03-60138),03-60138,NaN,NaN,03,271
2,03A265.htm,03A265,NaN,"Dameon Sean White, Applicant v. California","Court of Appeal of California, Fifth Appellate...",(F039132),F039132,NaN,NaN,03,265
3,03A503.htm,03A503,NaN,"James E. Reid, Applicant v. Page True, Warden",United States Court of Appeals for the Fourth ...,"(02-27, 03-2)","02-27, 03-2",NaN,NaN,03,503
4,03A259.htm,03A259,NaN,"Stephan A. Bitterman, Individually, and as Per...",Supreme Court of Florida,(SC03-1059),SC03-1059,NaN,NaN,03,259


# Confirm case numbers are bound between 1-2000 for all years

In [5]:
# Define valid bounds
lower_range = (1, 2000)

# Function to check if all case numbers fall within the allowed ranges
def is_year_valid(series):
    return series.dropna().apply(
        lambda x: lower_range[0] <= x <= lower_range[1]).all()

# Group by year and check validity
validity_by_year = scotus_df.groupby("year")["case_num"].apply(is_year_valid)
invalid_years = validity_by_year[~validity_by_year]

assert invalid_years.empty

# Check for years where the max for each range might have been cut off

In [6]:
upper_range = (1, 2000)

grouped = scotus_df.groupby("year")

upper_max = grouped.apply(lambda g: g.loc[g["case_num"].between(*upper_range), "case_num"].max())
upper_max

/var/folders/d9/3h7m7wc52kv6fgxmbyd8s0940000gn/T/ipykernel_83741/3779466430.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  upper_max = grouped.apply(lambda g: g.loc[g["case_num"].between(*upper_range), "case_num"].max())


year
01     861
02    1097
03    1065
04    1081
05    1236
06    1243
07    1044
08    1167
09    1268
10    1277
11    1256
12    1251
13    1286
14    1313
15    1303
16    1269
17    1425
18    1369
19    1071
20     178
21     876
22    1131
23    1177
24    1295
dtype: int64

# Confirm no missing docket numbers

In [7]:
assert len(scotus_df[scotus_df["docket_number"].isna()]) == 0

# Confirm no duplicate docket numbers

In [8]:
assert len(scotus_df[scotus_df.duplicated(subset=['docket_number'])]) == 0

# Confirm no missing case titles

In [9]:
assert len(scotus_df[scotus_df["case_title"].isnull()]) == 0

# Check for number of missing lower court name or lower court case numbers for each year

Manually verify a sample of those with lower court but without lower court case numbers is due to missing lower court case number info from scotus

In [10]:
scotus_df[(~scotus_df["lower_court"].isnull()) & (scotus_df["lower_court_case_numbers_raw"].isnull())]

,filename,docket_number,docket_date,case_title,lower_court,lower_court_case_numbers_raw,lower_court_case_numbers,lower_court_decision_date,lower_court_rehearing_denied_date,year,case_num
83,03A879.htm,03A879,NaN,"Margaret Bagley, Warden, Applicant v. Gregory ...",United States Court of Appeals for the Sixth C...,None,None,NaN,NaN,03,879
155,03A317.htm,03A317,NaN,"Robert Freeman, Applicant v. Pennsylvania","Supreme Court of Pennsylvania, Eastern District",None,None,NaN,NaN,03,317
860,03A247.htm,03A247,NaN,"Jose Martinez, Jr., Applicant v. United States",United States Court of Appeals for the Ninth C...,None,None,NaN,NaN,03,247
946,03A120.htm,03A120,NaN,"George T. Loebe, Sr., Applicant v. Glenn R. Fi...",Supreme Court of South Carolina,None,None,NaN,NaN,03,120
1800,04A1078.htm,04A1078,NaN,"United States, Applicant v. Sean Lamont Cromer",United States Court of Appeals for the Sixth C...,None,None,NaN,NaN,04,1078
2493,05A83.htm,05A83,NaN,"Donald J. Strable, Applicant v. South Carolina",Supreme Court of South Carolina,None,None,NaN,NaN,05,83
2853,05A920.htm,05A920,NaN,"Wilma Rowell-Poston, Applicant v. Florence Cou...",Supreme Court of South Carolina,None,None,NaN,NaN,05,920
2894,05A467.htm,05A467,NaN,"Shawn Paul Humphries, Applicant v. South Carolina",Supreme Court of South Carolina,None,None,NaN,NaN,05,467
3137,05A980.htm,05A980,NaN,"Janet Lutkewitte, Applicant v. Alberto R. Gonz...",United States Court of Appeals for the Distric...,None,None,NaN,NaN,05,980
3463,02A572.htm,02A572,NaN,"Kenneth Ford, Applicant v. Pennsylvania","Supreme Court of Pennsylvania, Eastern District",None,None,None,NaN,02,572


Confirm the only instance where we have the lower court case number but did not have the lower court was due to missing info from scotus

In [11]:
scotus_df[(scotus_df["lower_court"].isnull()) & (~scotus_df["lower_court_case_numbers_raw"].isnull())]

,filename,docket_number,docket_date,case_title,lower_court,lower_court_case_numbers_raw,lower_court_case_numbers,lower_court_decision_date,lower_court_rehearing_denied_date,year,case_num
25216,22A1115.json,22A1115,2023-06-23,"In Re Sealed Case, Applicant v.",None,(),,NaN,NaN,22,1115


Confirm majority of the cases with missing lower court information are cases "In Re"

In [12]:
subset = scotus_df[scotus_df["lower_court_case_numbers_raw"].isnull()]
starts_with_in_re = subset["case_title"].str.startswith("In Re")

num_in_re = starts_with_in_re.sum()
num_not_in_re = (~starts_with_in_re).sum()

print(f"Num Cases that start with 'In Re': {num_in_re} / {len(subset)}")
print(f"Num Cases that do NOT start with 'In Re': {num_not_in_re} / {len(subset)}")

Num Cases that start with 'In Re': 278 / 352
Num Cases that do NOT start with 'In Re': 74 / 352


Manually verify a sample of those without lower court and not In Re cases is due to missing lower court info from scotus

In [13]:
subset[~starts_with_in_re]

,filename,docket_number,docket_date,case_title,lower_court,lower_court_case_numbers_raw,lower_court_case_numbers,lower_court_decision_date,lower_court_rehearing_denied_date,year,case_num
83,03A879.htm,03A879,NaN,"Margaret Bagley, Warden, Applicant v. Gregory ...",United States Court of Appeals for the Sixth C...,None,None,NaN,NaN,03,879
155,03A317.htm,03A317,NaN,"Robert Freeman, Applicant v. Pennsylvania","Supreme Court of Pennsylvania, Eastern District",None,None,NaN,NaN,03,317
260,03A249.htm,03A249,NaN,"John Doe, Applicant v. United States",None,None,None,NaN,NaN,03,249
417,03A26.htm,03A26,NaN,"In re Riley D. Noel, Applicant v.",None,None,None,NaN,NaN,03,26
502,03A982.htm,03A982,NaN,"Daniel Benitez, Applicant v. John Mata",None,None,None,NaN,NaN,03,982
...,...,...,...,...,...,...,...,...,...,...,...
24825,22A431.json,22A431,2022-11-15,"Murray Hooper, Applicant v. David Shinn, et al.",None,None,None,NaN,NaN,22,431
24966,22A447.json,22A447,2022-11-21,"Bradly Cunningham, Applicant v. Oregon",None,None,None,NaN,NaN,22,447
25255,22A334.json,22A334,2022-10-20,"Oscar Stilley, Applicant v. United States",None,None,None,NaN,NaN,22,334
25346,22A439.json,22A439,2022-11-16,"Richard Fairchild, Applicant v. Jim Farris, W...",None,None,None,NaN,NaN,22,439


# Save the data for future use

In [14]:
scotus_df.to_json("data/scotus_data_A.json")

In [15]:
len(scotus_df)

25831